# 06 — Modèles finaux sur le jeu Spark complet (11 mois)

Entraîne et compare les modèles sur la couche **Gold** produite par le notebook 05 (`gold_ml_entrant.parquet`, trafic entrant, 8 fichiers), puis **exporte** le système retenu pour le scoring en flux et la plateforme.

| Étape | Contenu |
|---|---|
| Données | Gold Spark, trafic entrant, tri temporel ; fichier 008 (tempête d'alertes de 51 min) exclu par défaut |
| Découpage | temporel : ajustement 64 % → validation 16 % (seuils) → test 20 % (jamais utilisé pour choisir) |
| Modèles | Baseline, régression logistique, Random Forest, LightGBM, Isolation Forest (flux) ; TF-IDF + régression logistique (texte HTTP) ; **hybride** texte / flux |
| Évaluation | AUC-PR (principale), ROC-AUC, précision, rappel, F1 ; IC 95 % bootstrap par IP ; dérive par mois ; rappel par famille ; textes jamais vus |
| MLOps | une **seule** base MLflow, modèles enregistrés dans le **Model Registry** ; export `joblib` + prédictions + métriques pour la plateforme |

**Inputs Kaggle** : la sortie du notebook `05_spark_lakehouse` (contient `gold_ml_entrant.parquet`). *Accelerator* = None, *Internet* = On. Durée estimée : 30 à 60 min.

In [1]:
!pip install -q mlflow

## 0. Paramètres et chargement

In [2]:
import os, glob, json, re, time
import numpy as np, pandas as pd, joblib
import mlflow, mlflow.sklearn
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, precision_recall_curve)

SEED = 42
WORK = "/kaggle/working"
EXCLUDE_FILES = ["008"]        # tempête d'alertes de 51 min (voir notebook 05) ; [] pour la garder
RECALL_TARGET = 0.90           # seuil = le plus haut qui garde ce rappel sur la VALIDATION
N_BOOT = 200
os.makedirs(f"{WORK}/models", exist_ok=True)
T0 = time.time()

path = sorted(glob.glob("/kaggle/input/**/gold_ml_entrant.parquet", recursive=True))
assert path, "gold_ml_entrant.parquet introuvable : attacher la sortie du notebook 05 en Input"
df = pd.read_parquet(path[0])
print("Fichier :", path[0], "| lignes brutes :", len(df))
print("Lignes par fichier source :", df["source_file"].value_counts().sort_index().to_dict())
df = df[~df["source_file"].isin(EXCLUDE_FILES)]
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df = df.sort_values(["ts", "flow_id"], kind="mergesort").reset_index(drop=True)
df["text"] = df["text"].where(df["has_http"] == 1, None)

print(f"\nAprès exclusion de {EXCLUDE_FILES} : {len(df):,} alertes | {df['src_ip'].nunique():,} IP | "
      f"{df['ts'].min():%Y-%m-%d} → {df['ts'].max():%Y-%m-%d}")
print(f"Prévalence (label v1) : {df['label'].mean():.3f} | part avec requête HTTP : {df['has_http'].mean():.3f}")
monthly = df.groupby(df["ts"].dt.strftime("%Y-%m")).agg(alertes=("label", "size"), prevalence=("label", "mean"),
                                                         part_http=("has_http", "mean"), ip=("src_ip", "nunique"))
print(monthly.round(3).to_string())

Fichier : /kaggle/input/datasets/nawresbm2/suricata-gold/gold_ml_entrant.parquet | lignes brutes : 981662
Lignes par fichier source : {'001': 103621, '002': 102480, '003': 131780, '004': 152359, '005': 186802, '006': 138789, '007': 165619, '008': 212}

Après exclusion de ['008'] : 981,450 alertes | 28,049 IP | 2024-03-12 → 2025-02-11
Prévalence (label v1) : 0.279 | part avec requête HTTP : 0.839
         alertes  prevalence  part_http    ip
ts                                           
2024-03    44977       0.188      0.864  3050
2024-04    36741       0.319      0.699  4501
2024-05    39082       0.328      0.695  4764
2024-06    66510       0.405      0.836  4879
2024-07   101002       0.412      0.846  5127
2024-08    95009       0.202      0.844  5318
2024-09   153018       0.332      0.901  6798
2024-10   114905       0.192      0.826  6319
2024-11    75691       0.238      0.731  4907
2024-12    96157       0.216      0.837  4720
2025-01    92848       0.265      0.882  4846
202

## 1. Variables et découpage temporel
Mêmes 19 variables numériques + 4 catégorielles (one-hot) que le prototype. Les catégories sont apprises sur le **train** uniquement. `sig`, `sig_category`, `sig_severity`, `label_v2` ne sont **jamais** des variables (ils définissent ou trahissent le label).

In [3]:
NUM = ["dest_port", "pkts_toserver", "pkts_toclient", "bytes_toserver", "bytes_toclient", "age", "bytes_ratio",
       "pkts_total", "hour", "dow", "ip_prev_alerts", "ip_gap_s", "hist_flows", "hist_alerted", "hist_alert_rate",
       "is_new_host", "prev1h_flows", "prev1h_alerted", "prev1h_ports"]
CAT = ["proto", "app_proto", "state", "reason"]

def make_X(d, cols=None):
    X = pd.get_dummies(d[NUM + CAT], columns=CAT, dummy_na=True).astype("float32").fillna(0)
    X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in X.columns]
    return X if cols is None else X.reindex(columns=cols, fill_value=0)

n = len(df)
cut = int(n * 0.8)                 # train | test
vcut = int(cut * 0.8)              # ajustement | validation (dans le train)
i_fit, i_val, i_tr, i_te = np.arange(vcut), np.arange(vcut, cut), np.arange(cut), np.arange(cut, n)

FEATS = list(make_X(df.iloc[:cut]).columns)
X = make_X(df, FEATS)
y = df["label"].values.astype(int)
http = (df["has_http"] == 1).values
assert not any(c.startswith(("sig", "label", "severity", "category")) for c in FEATS)

for name, idx in [("ajustement", i_fit), ("validation", i_val), ("test", i_te)]:
    print(f"{name:11s} {len(idx):>9,} lignes | {df['ts'].iloc[idx[0]]:%Y-%m-%d} → {df['ts'].iloc[idx[-1]]:%Y-%m-%d} | "
          f"prévalence {y[idx].mean():.3f} | HTTP {http[idx].mean():.2f}")
print("Variables :", len(FEATS))

ajustement    628,128 lignes | 2024-03-12 → 2024-10-22 | prévalence 0.301 | HTTP 0.84
validation    157,032 lignes | 2024-10-22 → 2024-12-22 | prévalence 0.239 | HTTP 0.77
test          196,290 lignes | 2024-12-22 → 2025-02-11 | prévalence 0.242 | HTTP 0.90
Variables : 35


## 2. Fonctions d'évaluation

In [4]:
def thr_for_recall(yv, pv, target=RECALL_TARGET):
    prec, rec, thr = precision_recall_curve(yv, pv)
    ok = np.where(rec[:-1] >= target)[0]
    return float(thr[ok[-1]]) if len(ok) else 0.5

def scores(yt, p, thr=0.5):
    pr = (p >= thr).astype(int)
    return {"auc_pr": average_precision_score(yt, p), "roc_auc": roc_auc_score(yt, p),
            "precision": precision_score(yt, pr, zero_division=0), "recall": recall_score(yt, pr, zero_division=0),
            "f1": f1_score(yt, pr, zero_division=0), "threshold": thr}

ips_te = df["src_ip"].values[i_te]
codes_te, _ = pd.factorize(ips_te)
groups_te = pd.Series(np.arange(len(i_te))).groupby(codes_te).apply(np.array).values

def boot_ci(p, p_ref=None, n_boot=N_BOOT):
    # IC 95 % en rééchantillonnant les IP (les alertes d'une même IP sont corrélées)
    rng, a, d = np.random.RandomState(SEED), [], []
    yt = y[i_te]
    for _ in range(n_boot):
        ii = np.concatenate(groups_te[rng.randint(0, len(groups_te), len(groups_te))])
        if yt[ii].sum() == 0:
            continue
        s = average_precision_score(yt[ii], p[ii]); a.append(s)
        if p_ref is not None:
            d.append(s - average_precision_score(yt[ii], p_ref[ii]))
    out = {"ci_low": float(np.percentile(a, 2.5)), "ci_high": float(np.percentile(a, 97.5))}
    if p_ref is not None:
        out |= {"diff_mean": float(np.mean(d)), "diff_low": float(np.percentile(d, 2.5)), "diff_high": float(np.percentile(d, 97.5))}
    return out

results, test_preds = {}, {}
print("Prévalence du test (AUC-PR d'un modèle sans information) :", round(y[i_te].mean(), 3))

Prévalence du test (AUC-PR d'un modèle sans information) : 0.242


## 3. Modèles sur les variables de flux
Pour LightGBM (modèle déployé) : seuil choisi sur la **validation** avec un modèle entraîné sur l'ajustement, puis modèle final réentraîné sur **tout le train** (ajustement + validation) et évalué une seule fois sur le test. Les autres modèles servent de comparaison (seuil 0,5).

In [5]:
def fit_eval(name, model, family, Xtr, ytr, Xte, thr=0.5, proba=True):
    t0 = time.time()
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1] if proba else model(Xte)
    results[name] = {"family": family, **scores(y[i_te], p, thr), "train_s": round(time.time() - t0, 1)}
    test_preds[name] = p
    print(f"{name:28s} AUC-PR {results[name]['auc_pr']:.3f} | ROC {results[name]['roc_auc']:.3f} | "
          f"P {results[name]['precision']:.3f} R {results[name]['recall']:.3f} | {results[name]['train_s']:.0f} s")
    return model

Xtr, ytr, Xte = X.iloc[i_tr], y[i_tr], X.iloc[i_te]
fit_eval("Baseline (prior)", DummyClassifier(strategy="prior"), "Reference", Xtr, ytr, Xte)
fit_eval("Logistic regression", make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
         "Linear", Xtr, ytr, Xte)
fit_eval("Random Forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced_subsample",
                                                 n_jobs=-1, random_state=SEED), "Bagging", Xtr, ytr, Xte)

# LightGBM : seuil sur la validation, puis réentraînement sur tout le train
lgbm_params = dict(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1)
lg_a = LGBMClassifier(**lgbm_params).fit(X.iloc[i_fit], y[i_fit])
pv_flow = lg_a.predict_proba(X.iloc[i_val])[:, 1]
THR_FLOW = thr_for_recall(y[i_val], pv_flow)
print(f"Seuil LightGBM (validation, rappel >= {RECALL_TARGET}) : {THR_FLOW:.3f} | AUC-PR validation {average_precision_score(y[i_val], pv_flow):.3f}")
lgbm = fit_eval("LightGBM (flow)", LGBMClassifier(**lgbm_params), "Gradient boosting", Xtr, ytr, Xte, thr=THR_FLOW)

# Isolation Forest (non supervisé) : entraîné sur un échantillon du train, score = anomalie
rng = np.random.RandomState(SEED)
samp = rng.choice(i_tr, size=min(200_000, len(i_tr)), replace=False)
iso = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X.iloc[samp])
p_iso = -iso.score_samples(Xte)
p_iso = (p_iso - p_iso.min()) / (p_iso.max() - p_iso.min() + 1e-12)
results["Isolation Forest"] = {"family": "Anomaly detection", **scores(y[i_te], p_iso, float(np.quantile(p_iso, 1 - y[i_tr].mean()))),
                               "train_s": None}
test_preds["Isolation Forest"] = p_iso
print(f"{'Isolation Forest':28s} AUC-PR {results['Isolation Forest']['auc_pr']:.3f} | ROC {results['Isolation Forest']['roc_auc']:.3f}")

Baseline (prior)             AUC-PR 0.242 | ROC 0.500 | P 0.000 R 0.000 | 0 s
Logistic regression          AUC-PR 0.666 | ROC 0.811 | P 0.525 R 0.622 | 11 s
Random Forest                AUC-PR 0.804 | ROC 0.916 | P 0.538 R 0.931 | 155 s
Seuil LightGBM (validation, rappel >= 0.9) : 0.478 | AUC-PR validation 0.906
LightGBM (flow)              AUC-PR 0.783 | ROC 0.913 | P 0.535 R 0.939 | 15 s
Isolation Forest             AUC-PR 0.269 | ROC 0.566


## 4. Modèle texte HTTP et système hybride
TF-IDF sur n-grammes de caractères (2 à 5) de `méthode URL || user-agent` + régression logistique, sur les alertes qui ont une requête HTTP. **Hybride** : texte si la requête existe, sinon LightGBM. Seuil de l'hybride choisi sur la validation.

In [6]:
def text_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=200_000, min_df=3,
                                  sublinear_tf=True, lowercase=False, dtype=np.float32)),
        ("lr", LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, solver="liblinear")),
    ])
texts = df["text"].fillna("").values

# Étape A : modèles entraînés sur l'ajustement -> seuil de l'hybride sur la validation
t0 = time.time()
fa = i_fit[http[i_fit]]
txt_a = text_model().fit(texts[fa], y[fa])
pv_h = pv_flow.copy()
vh = http[i_val]
pv_h[vh] = txt_a.predict_proba(texts[i_val][vh])[:, 1]
THR_HYB = thr_for_recall(y[i_val], pv_h)
print(f"Étape A en {(time.time() - t0) / 60:.1f} min | AUC-PR validation hybride {average_precision_score(y[i_val], pv_h):.3f} "
      f"| seuil hybride {THR_HYB:.3f}")

# Étape B : modèle texte final sur tout le train, évalué sur le test
t0 = time.time()
ft = i_tr[http[i_tr]]
txt = text_model().fit(texts[ft], y[ft])
te_h = http[i_te]
p_text = np.full(len(i_te), np.nan)
p_text[te_h] = txt.predict_proba(texts[i_te][te_h])[:, 1]
p_hyb = np.where(te_h, p_text, test_preds["LightGBM (flow)"])
print(f"Étape B en {(time.time() - t0) / 60:.1f} min | vocabulaire : {len(txt.named_steps['tfidf'].vocabulary_):,} n-grammes")

results["TF-IDF + LR (HTTP text only)"] = {"family": "Classical NLP", **scores(y[i_te][te_h], p_text[te_h], THR_HYB),
                                          "train_s": None, "perimeter": "test alerts with HTTP request"}
results["Hybrid (text / flow)"] = {"family": "Hybrid", **scores(y[i_te], p_hyb, THR_HYB), "train_s": None}
test_preds["Hybrid (text / flow)"] = p_hyb
for k in ["TF-IDF + LR (HTTP text only)", "Hybrid (text / flow)"]:
    r = results[k]
    print(f"{k:28s} AUC-PR {r['auc_pr']:.3f} | ROC {r['roc_auc']:.3f} | P {r['precision']:.3f} R {r['recall']:.3f}")

Étape A en 5.2 min | AUC-PR validation hybride 0.995 | seuil hybride 0.950
Étape B en 6.9 min | vocabulaire : 200,000 n-grammes
TF-IDF + LR (HTTP text only) AUC-PR 0.981 | ROC 0.989 | P 1.000 R 0.750
Hybrid (text / flow)         AUC-PR 0.985 | ROC 0.991 | P 0.999 R 0.790


## 5. Comparaison, intervalles de confiance et robustesse

In [7]:
ci_lgbm = boot_ci(test_preds["LightGBM (flow)"])
ci_hyb = boot_ci(p_hyb, p_ref=test_preds["LightGBM (flow)"])
results["LightGBM (flow)"] |= ci_lgbm
results["Hybrid (text / flow)"] |= ci_hyb
res = pd.DataFrame(results).T
cols = ["family", "auc_pr", "ci_low", "ci_high", "roc_auc", "precision", "recall", "f1", "threshold"]
print(f"Test : {len(i_te):,} alertes | prévalence {y[i_te].mean():.3f}")
print(res[[c for c in cols if c in res]].sort_values("auc_pr", ascending=False).to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\nGain hybride − LightGBM : {ci_hyb['diff_mean']:.3f} IC 95 % [{ci_hyb['diff_low']:.3f} ; {ci_hyb['diff_high']:.3f}]")

# Dérive : AUC-PR par mois du test
mo = df["ts"].iloc[i_te].dt.strftime("%Y-%m").values
drift = pd.DataFrame([{"mois": m, "alertes": int((mo == m).sum()), "prevalence": y[i_te][mo == m].mean(),
                       "auc_pr_lgbm": average_precision_score(y[i_te][mo == m], test_preds["LightGBM (flow)"][mo == m]),
                       "auc_pr_hybride": average_precision_score(y[i_te][mo == m], p_hyb[mo == m])}
                      for m in sorted(set(mo)) if 0 < y[i_te][mo == m].sum() < (mo == m).sum()])
print("\nDérive par mois (test) :"); print(drift.round(3).to_string(index=False))

# Textes jamais vus dans le train et rappel par famille
seen = set(texts[ft])
new_txt = te_h & ~np.isin(texts[i_te], list(seen))
print(f"\nTextes du test jamais vus dans le train : {new_txt[te_h].mean():.1%} | AUC-PR hybride sur ces alertes : "
      f"{average_precision_score(y[i_te][new_txt], p_hyb[new_txt]):.3f}" if y[i_te][new_txt].sum() else "")
fam = pd.DataFrame({"sig": df["sig"].values[i_te], "y": y[i_te],
                    "hybride": (p_hyb >= THR_HYB).astype(int), "lgbm": (test_preds["LightGBM (flow)"] >= THR_FLOW).astype(int)})
print("\nRappel par famille positive (test, top 12) :")
print(fam[fam["y"] == 1].groupby("sig").agg(n=("y", "size"), rappel_lgbm=("lgbm", "mean"), rappel_hybride=("hybride", "mean"))
         .sort_values("n", ascending=False).head(12).round(2).to_string())
print("\nFaux positifs de l'hybride par famille (top 6) :")
print(fam[(fam["y"] == 0) & (fam["hybride"] == 1)]["sig"].value_counts().head(6).to_string())

Test : 196,290 alertes | prévalence 0.242
                                         family auc_pr ci_low ci_high roc_auc precision recall    f1 threshold
Hybrid (text / flow)                     Hybrid  0.985  0.978   0.992   0.991     0.999  0.790 0.882     0.950
TF-IDF + LR (HTTP text only)      Classical NLP  0.981    NaN     NaN   0.989     1.000  0.750 0.857     0.950
Random Forest                           Bagging  0.804    NaN     NaN   0.916     0.538  0.931 0.682     0.500
LightGBM (flow)               Gradient boosting  0.783  0.662   0.952   0.913     0.535  0.939 0.682     0.478
Logistic regression                      Linear  0.666    NaN     NaN   0.811     0.525  0.622 0.569     0.500
Isolation Forest              Anomaly detection  0.269    NaN     NaN   0.566     0.264  0.314 0.287     0.329
Baseline (prior)                      Reference  0.242    NaN     NaN   0.500     0.000  0.000 0.000     0.500

Gain hybride − LightGBM : 0.191 IC 95 % [0.036 ; 0.318]

Dérive par m

## 6. MLflow : une seule base, runs et Model Registry
Tous les modèles du notebook dans l'expérience `suricata-final`. LightGBM et le modèle texte sont enregistrés dans le **Model Registry** (`suricata-lgbm-flow`, `suricata-tfidf-text`) : c'est la « gestion des modèles » demandée.

In [10]:
# Nouvelle base (l'ancienne connexion vers mlflow.db est restée ouverte dans cette session)
DB = f"{WORK}/mlflow_final.db"
ART = f"{WORK}/mlartifacts_final"
mlflow.set_tracking_uri(f"sqlite:///{DB}")
mlflow.set_registry_uri(f"sqlite:///{DB}")
EXP = "suricata-final"
if mlflow.get_experiment_by_name(EXP) is None:
    mlflow.create_experiment(EXP, artifact_location=ART)
mlflow.set_experiment(EXP)

def log_model(m, registered):
    # MLflow 3 sérialise par défaut avec skops, qui refuse les objets LightGBM : on force cloudpickle
    kw = dict(serialization_format="cloudpickle", registered_model_name=registered)
    try:
        mlflow.sklearn.log_model(sk_model=m, name="model", **kw)
    except TypeError:
        mlflow.sklearn.log_model(sk_model=m, artifact_path="model", **kw)

common = {"data": "Gold Spark, trafic entrant, 8 fichiers", "excluded_files": ",".join(EXCLUDE_FILES) or "aucun",
          "split": "temporel 64/16/20", "n_train": len(i_tr), "n_test": len(i_te), "label": "v1", "n_features": len(FEATS)}
for name, r in results.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_params(common | {"model": name, "family": r["family"]})
        mlflow.log_metrics({k: float(v) for k, v in r.items() if isinstance(v, (int, float, np.floating)) and v is not None})
        if name == "LightGBM (flow)":
            mlflow.log_params({f"lgbm_{k}": v for k, v in lgbm_params.items()})
            log_model(lgbm, "suricata-lgbm-flow")
        if name == "TF-IDF + LR (HTTP text only)":
            log_model(txt, "suricata-tfidf-text")
        if name == "Hybrid (text / flow)":
            mlflow.log_params({"rule": "texte si requête HTTP sinon LightGBM", "threshold_hybrid": round(THR_HYB, 4)})
            mlflow.log_text(drift.to_csv(index=False), "drift_by_month.csv")
print("Runs enregistrés :", len(results), "| base :", DB)

2026/11/23 14:51:51 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/23 14:51:52 INFO mlflow.store.db.utils: Updating database tables
2026/11/23 14:51:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'suricata-lgbm-flow'.
Created version '1' of model 'suricata-lgbm-flow'.
2026/11/23 14:52:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more info

Runs enregistrés : 7 | base : /kaggle/working/mlflow_final.db


Successfully registered model 'suricata-tfidf-text'.
Created version '1' of model 'suricata-tfidf-text'.


## 7. Export : modèles, prédictions, métriques pour la plateforme
- `models/lgbm_flow.joblib` et `models/tfidf_text.joblib` : utilisés par le service de scoring en flux.
- `predictions_test.parquet` : une ligne par alerte du test, avec tout ce que le dashboard affiche (fichier unique pour `import_results.py`).
- `model_results_final.json` : métriques au format de la page **Models** de la plateforme.

In [11]:
joblib.dump({"model": lgbm, "features": FEATS, "num": NUM, "cat": CAT, "threshold": THR_FLOW,
             "trained_until": str(df["ts"].iloc[cut - 1])}, f"{WORK}/models/lgbm_flow.joblib")
joblib.dump({"model": txt, "threshold": THR_HYB, "text_format": "METHOD URL || USER_AGENT"}, f"{WORK}/models/tfidf_text.joblib")
json.dump({"rule": "text model if an HTTP request is linked to the alert, otherwise LightGBM on flow features",
           "threshold": THR_HYB, "threshold_flow": THR_FLOW, "label": "v1", "features": FEATS},
          open(f"{WORK}/models/hybrid_config.json", "w"), indent=1)

t = df.iloc[i_te]
pred = pd.DataFrame({
    "flow_id": t["flow_id"].values, "ts": t["ts"].values, "src_ip": t["src_ip"].values, "dest_ip": t["dest_ip"].values,
    "dest_port": t["dest_port"].values, "proto": t["proto"].values, "app_proto": t["app_proto"].values,
    "label": y[i_te], "sig": t["sig"].values, "p_flux": test_preds["LightGBM (flow)"], "p_texte": p_text,
    "p_final": p_hyb, "source": np.where(te_h, "texte_http", "flux"), "risk": np.round(100 * p_hyb, 1),
    "method": t["method"].values, "url": t["url"].values, "user_agent": t["user_agent"].values, "status": t["status"].values,
})
pred["threshold"] = THR_HYB
pred.to_parquet(f"{WORK}/predictions_test.parquet", index=False)

per_test = f"Test, inbound alerts ({len(i_te):,}), {t['ts'].min():%b %Y} → {t['ts'].max():%b %Y}"
per_txt = f"Test, inbound alerts with HTTP request ({int(te_h.sum()):,})"
role = {"Hybrid (text / flow)": "Production system", "LightGBM (flow)": "Fallback when no HTTP text",
        "TF-IDF + LR (HTTP text only)": "Text model in production", "Isolation Forest": "Disabled (weight 0)",
        "Baseline (prior)": "Reference"}
out = []
for name, r in results.items():
    out.append({"name": name, "family": r["family"], "perimeter": per_txt if "only" in name else per_test,
                "aucPr": round(float(r["auc_pr"]), 3), "ciLow": round(r["ci_low"], 3) if "ci_low" in r else None,
                "ciHigh": round(r["ci_high"], 3) if "ci_high" in r else None,
                "precision": round(float(r["precision"]), 3), "recall": round(float(r["recall"]), 3),
                "rocAuc": round(float(r["roc_auc"]), 3), "role": role.get(name, "Comparison"),
                "note": (f"Gain over LightGBM: {ci_hyb['diff_mean']:+.3f} [{ci_hyb['diff_low']:.3f}; {ci_hyb['diff_high']:.3f}]"
                         if name == "Hybrid (text / flow)" else "")})
json.dump(out, open(f"{WORK}/model_results_final.json", "w"), indent=1)
res.to_csv(f"{WORK}/resultats_finaux.csv")

print("Exporté :", sorted(os.listdir(f"{WORK}/models")), "+ predictions_test.parquet, model_results_final.json, resultats_finaux.csv")
print(f"Durée totale : {(time.time() - T0) / 60:.1f} min")

Exporté : ['hybrid_config.json', 'lgbm_flow.joblib', 'tfidf_text.joblib'] + predictions_test.parquet, model_results_final.json, resultats_finaux.csv
Durée totale : 29.7 min


In [12]:
from IPython.display import FileLink, display
for f in ["models/lgbm_flow.joblib", "models/tfidf_text.joblib", "models/hybrid_config.json",
          "predictions_test.parquet", "model_results_final.json", "resultats_finaux.csv", "mlflow_final.db"]:
    display(FileLink(f))

/kaggle/working/models/lgbm_flow.joblib

/kaggle/working/models/tfidf_text.joblib

/kaggle/working/models/hybrid_config.json

/kaggle/working/predictions_test.parquet

/kaggle/working/model_results_final.json

/kaggle/working/resultats_finaux.csv

/kaggle/working/mlflow_final.db